# 06. Query-based: RT-DETR

In [ ]:
import subprocess
import sys
from pathlib import Path

if Path("../src/vod").exists():
    sys.path.insert(0, "../src")
else:
    repo = Path("/tmp/repo")  # outside /kaggle/working so it is not saved as output
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/mohinska/ai_hw2_MultipleObjectVideoDetection", str(repo)], check=True)
    sys.path.insert(0, str(repo / "src"))

In [ ]:
!pip install -q ultralytics faster-coco-eval

In [ ]:
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from ultralytics import settings

from vod import CLASSES
from vod.data import find_data_root, load_split
from vod.protocol import PROTOCOL
from vod.submission import write_submission
from vod.tv import comparison_tables, completed_runs
from vod.variants import collect_view, yolo_detector
from vod.yolo import evaluate_yolo, load_ultralytics, train_yolo, write_data_yaml, write_dataset

DATA = find_data_root()
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../outputs")
WORK.mkdir(parents=True, exist_ok=True)
TMP = Path("/tmp/rtdetr_data") if Path("/kaggle/working").exists() else WORK / "rtdetr_data"
ROOTS = (WORK, Path("/kaggle/input"))
settings.update({"wandb": False})

## Data

In [ ]:
train_frames, train_targets = load_split(DATA / "train", step=PROTOCOL["train_step"])
val_frames, val_targets = load_split(DATA / "val", step=1)
VAL_GT = DATA / "val" / "annotations_coco.json"
yaml_path = write_data_yaml(TMP / "data.yaml", write_dataset(train_frames, train_targets, TMP / "train"),
                            write_dataset(val_frames.iloc[::PROTOCOL["val_step_epoch"]], val_targets, TMP / "val"))

## Objects per frame vs the 300 queries

In [ ]:
counts = pd.DataFrame({
    split: pd.Series([len(t[0]) for t in targets.values()]).describe(percentiles=[0.5, 0.95, 0.99])
    for split, targets in [("train", train_targets), ("val", val_targets)]}).round(1)
counts.loc["share_over_300"] = [np.mean([len(t[0]) > 300 for t in targets.values()]).round(4)
                                for targets in (train_targets, val_targets)]
counts

## Experiments

In [ ]:
BASE = dict(weights="rtdetr-l.pt", imgsz=PROTOCOL["imgsz"], epochs=10, batch=4, close_mosaic=2,
            optimizer="AdamW", lr0=1e-4, weight_decay=1e-4, cos_lr=True)
EXPERIMENTS = {
    "rtdetr_default": BASE,
    "rtdetr_lr3e4": {**BASE, "lr0": 3e-4},
    "rtdetr_giou4": {**BASE, "loss_gain": {"class": 1, "bbox": 5, "giou": 4}},
    # same 7 h budget: Ultralytics fits the epochs and the lr schedule into `time` hours (~25 epochs of L);
    # close_mosaic 5 keeps the last 20% of epochs without mosaic, as 2 of 10 above
    "rtdetr_lr3e4_7h": {**BASE, "lr0": 3e-4, "time": 7, "close_mosaic": 5},
    "rtdetr_x_7h": {**BASE, "weights": "rtdetr-x.pt", "lr0": 3e-4, "batch": 2, "time": 7, "close_mosaic": 5},
}
RUN = ["rtdetr_lr3e4_7h"]  # ~8 h per run, so one per account: the other one runs ["rtdetr_x_7h"]

## Train and evaluate (last epoch, every val frame)

In [ ]:
for name in RUN:
    if name in completed_runs(ROOTS):
        print(name, "done")
        continue
    print(name, "training")
    run_dir = train_yolo(name, EXPERIMENTS[name], yaml_path, WORK)
    m = evaluate_yolo(name, EXPERIMENTS[name], run_dir, val_frames, VAL_GT)
    print({k: round(m[k], 4) for k in ["mAP", "AP50", "AP75", "AP_small"]}, m["ms_per_image"], "ms/image")
    torch.cuda.empty_cache()

## Comparison with every finished run

In [ ]:
runs = completed_runs(ROOTS)
main, per_class, per_clip = comparison_tables(runs, CLASSES)
display(main)
display(per_class)
per_clip

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for n, m in runs.items():
    if n in EXPERIMENTS and m.get("history") is not None:
        ax.plot(m["history"]["epoch"], m["history"]["mAP"], marker="o", label=n)
ax.set(xlabel="epoch", ylabel="val mAP (every 10th frame)", title="Validation mAP during training")
ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig(WORK / "rtdetr_map_curves.png", dpi=110)

## Test predictions with the best RT-DETR

In [ ]:
ours = main[main.index.isin(EXPERIMENTS)]
BEST = ours["mAP"].idxmax()
print("best:", BEST, "| val mAP", ours.loc[BEST, "mAP"])
weights = next(p for root in ROOTS if root.exists() for p in root.rglob(f"{BEST}/weights/last.pt"))
test = pd.read_csv(DATA / "test_images.csv")
test_paths = [str(DATA / f) for f in test["file_name"]]
dets = collect_view(yolo_detector(load_ultralytics(weights), imgsz=PROTOCOL["imgsz"]), test_paths, "full")
dets["image_id"] = dets.pop("file_name").map(dict(zip(test_paths, test["image_id"])))
sub = write_submission(dets, DATA / "test_images.csv", WORK / f"submission_{BEST}.csv")
sub.head()

In [ ]:
shutil.rmtree(TMP, ignore_errors=True)